# RAG on "Attention Is All You Need" — working Colab notebook

A clean, working RAG pipeline over the Transformer paper, built with battle-tested libraries (LangChain + Chroma + OpenAI) instead of AutoRAG. Everything runs end-to-end in Colab.

**What you get:**
- PDF parse + chunk + embed + index
- Retrieval over the indexed corpus
- LLM generation grounded in retrieved passages
- Synthetic QA evaluation set generated from the corpus
- Retrieval metrics (recall@k, MRR, hit-rate)
- Generation metrics (LLM-as-judge correctness)
- Live Q&A interface against the final pipeline

**Cost:** ~5–10 cents on `gpt-4o-mini` for the whole notebook.

## 1. Install dependencies

In [1]:
!pip install -q langchain langchain-community langchain-openai langchain-chroma \
    chromadb openai pypdf tiktoken pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69

## 2. Set the OpenAI key

Defensive — works whether or not Colab Secrets is configured.

In [2]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste OpenAI key: ")

assert os.environ["OPENAI_API_KEY"].startswith("sk-"), "Key not loaded"
print("OpenAI key loaded.")

OpenAI key loaded.


## 3. Download the paper

In [3]:
!mkdir -p data
!wget -q -O data/attention.pdf https://arxiv.org/pdf/1706.03762.pdf
!ls -lh data/attention.pdf

-rw-r--r-- 1 root root 2.2M Apr 12  2024 data/attention.pdf


## 4. Load + chunk the PDF

LangChain's `PyPDFLoader` + `RecursiveCharacterTextSplitter` — the standard, battle-tested approach. 800-character chunks with 100-char overlap (good middle-ground for technical papers).

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("data/attention.pdf")
pages = loader.load()
print(f"Loaded {len(pages)} pages")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks")
print(f"\nFirst chunk preview:\n{chunks[0].page_content[:300]}...")

Loaded 15 pages
Split into 64 chunks

First chunk preview:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par...


## 5. Embed + index in Chroma

OpenAI `text-embedding-3-small` (1536-dim, fast, cheap) → in-memory Chroma. No persistence required for a notebook session.

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Add a chunk_id to each doc so we can track ground-truth retrieval later
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="attention",
)
print(f"Indexed {len(chunks)} chunks into Chroma")

# Sanity-check: do a similarity search
test = vectorstore.similarity_search("multi-head attention", k=2)
print(f"\nTop result for 'multi-head attention':")
print(test[0].page_content[:300])

Indexed 64 chunks into Chroma

Top result for 'multi-head attention':
output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows the model to jointly attend to information from different representation
subspaces at different positions. With a single attention head, averaging inhib


## 6. Build the RAG chain

Retriever (top-3) → prompt → `gpt-4o-mini` → answer. LangChain Expression Language (LCEL) keeps it simple and inspectable.

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_template(
    """Use the passages below to answer the question. Be concise and accurate.
If the passages don't contain the answer, say you don't know.

Passages:
{context}

Question: {question}

Answer:"""
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Quick smoke test
answer = rag_chain.invoke("What is multi-head attention?")
print("Q: What is multi-head attention?")
print(f"A: {answer}")

Q: What is multi-head attention?
A: Multi-head attention is a mechanism that allows a model to jointly attend to information from different representation subspaces at various positions. It consists of multiple attention layers (or heads) running in parallel, where each head computes attention using its own set of projections for queries, keys, and values. The outputs of these heads are concatenated and projected to produce the final attention values. This approach enables the model to capture diverse aspects of the input data while maintaining a similar computational cost to single-head attention with full dimensionality.


## 7. Generate a synthetic QA evaluation set

Sample 10 random chunks → ask `gpt-4o-mini` to write one factoid question per chunk where the chunk is the ground-truth source. Same idea as AutoRAG's `factoid_query_gen`, but ~30 lines and reliably runs.

In [8]:
import random, json
from langchain_core.prompts import ChatPromptTemplate

random.seed(0)
n_qa = 10
sampled_chunks = random.sample(chunks, n_qa)

qa_prompt = ChatPromptTemplate.from_template(
    """Read this passage from a research paper and write ONE factoid question that is answered by it.
Output JSON with two keys: "question" (string) and "answer" (string, 1-2 sentences from the passage).
Output ONLY the JSON object, no markdown fencing.

Passage:
{passage}"""
)

qa_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
qa_chain = qa_prompt | qa_llm | StrOutputParser()

qa_set = []
for i, chunk in enumerate(sampled_chunks):
    raw = qa_chain.invoke({"passage": chunk.page_content})
    # Strip any accidental markdown fencing
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        parsed = json.loads(raw)
        qa_set.append({
            "qid": i,
            "question": parsed["question"],
            "answer": parsed["answer"],
            "gt_chunk_id": chunk.metadata["chunk_id"],
            "gt_passage": chunk.page_content,
        })
    except Exception as e:
        print(f"Skipping QA #{i}: {e}")

print(f"\nGenerated {len(qa_set)} QA pairs\n")
for qa in qa_set[:3]:
    print(f"Q{qa['qid']}: {qa['question']}")
    print(f"   gt_answer: {qa['answer'][:120]}...")
    print()


Generated 10 QA pairs

Q0: Who are the authors of the paper titled 'Long short-term memory'?
   gt_answer: Sepp Hochreiter and Jürgen Schmidhuber are the authors of the paper titled 'Long short-term memory'....

Q1: Who are the authors of the paper titled 'Learning phrase representations using rnn encoder-decoder for statistical machine translation'?
   gt_answer: Kyunghyun Cho, Bart van Merrienboer, Caglar Gulcehre, Fethi Bougares, Holger Schwenk, and Yoshua Bengio....

Q2: Who are the authors of the paper titled 'Sequence to sequence learning with neural networks'?
   gt_answer: Ilya Sutskever, Oriol Vinyals, and Quoc VV Le....



## 8. Retrieval evaluation

For each QA pair, retrieve top-k chunks and check whether the ground-truth chunk is in there. Computes:
- **Hit Rate @ k** — fraction of queries where the gt chunk appears in top-k
- **MRR @ k** — mean reciprocal rank of the gt chunk
- **Recall @ k** — same as hit rate when there's one gt chunk per query

In [9]:
import pandas as pd

def evaluate_retrieval(qa_set, k=3):
    rows = []
    for qa in qa_set:
        retrieved = vectorstore.similarity_search(qa["question"], k=k)
        retrieved_ids = [d.metadata["chunk_id"] for d in retrieved]
        gt = qa["gt_chunk_id"]

        hit = gt in retrieved_ids
        rank = retrieved_ids.index(gt) + 1 if hit else None
        rr = 1.0 / rank if rank else 0.0

        rows.append({
            "qid": qa["qid"],
            "question": qa["question"][:80],
            "hit@k": hit,
            "rank": rank,
            "rr": rr,
        })
    return pd.DataFrame(rows)

ret_df = evaluate_retrieval(qa_set, k=3)
print(ret_df.to_string(index=False))
print(f"\nHit Rate @ 3: {ret_df['hit@k'].mean():.2%}")
print(f"MRR       @ 3: {ret_df['rr'].mean():.3f}")

 qid                                                                         question  hit@k  rank       rr
   0                Who are the authors of the paper titled 'Long short-term memory'?  False   NaN 0.000000
   1 Who are the authors of the paper titled 'Learning phrase representations using r   True   1.0 1.000000
   2 Who are the authors of the paper titled 'Sequence to sequence learning with neur   True   2.0 0.500000
   3 How do self-attention layers compare to recurrent layers in terms of connecting    True   3.0 0.333333
   4                            What BLEU score did the model achieve after training?   True   3.0 0.333333
   5 What is the variance of the dot product q · k when the components of q and k are   True   1.0 1.000000
   6                                   What optimizer was used in the research paper?   True   1.0 1.000000
   7                                 How many steps were the base models trained for?   True   1.0 1.000000
   8 What are the three desi

## 9. Generation evaluation (LLM-as-judge)

Run the full RAG chain on each question, then ask a separate LLM to judge whether the generated answer matches the ground-truth answer. Returns boolean per question + overall accuracy.

In [10]:
judge_prompt = ChatPromptTemplate.from_template(
    """You are an expert grader. Given a question, a ground-truth answer, and a predicted answer, decide if the prediction is semantically correct (captures the key facts of the ground truth).

Output ONLY one word: YES or NO.

Question: {question}
Ground truth: {gt}
Prediction: {pred}"""
)

judge_chain = judge_prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0) | StrOutputParser()

rows = []
for qa in qa_set:
    pred = rag_chain.invoke(qa["question"])
    verdict = judge_chain.invoke({
        "question": qa["question"],
        "gt": qa["answer"],
        "pred": pred,
    }).strip().upper()
    rows.append({
        "qid": qa["qid"],
        "question": qa["question"][:80],
        "prediction": pred[:120],
        "correct": verdict.startswith("YES"),
    })

gen_df = pd.DataFrame(rows)
print(gen_df.to_string(index=False))
print(f"\nGeneration accuracy: {gen_df['correct'].mean():.2%}")

 qid                                                                         question                                                                                                               prediction  correct
   0                Who are the authors of the paper titled 'Long short-term memory'?                                                                                                            I don't know.    False
   1 Who are the authors of the paper titled 'Learning phrase representations using r The authors of the paper titled 'Learning phrase representations using rnn encoder-decoder for statistical machine trans     True
   2 Who are the authors of the paper titled 'Sequence to sequence learning with neur The authors of the paper titled 'Sequence to sequence learning with neural networks' are Ilya Sutskever, Oriol Vinyals,      True
   3 How do self-attention layers compare to recurrent layers in terms of connecting  Self-attention layers connect all positions with a

## 10. Combined results

In [11]:
print("=" * 60)
print("RAG pipeline evaluation summary")
print("=" * 60)
print(f"  Corpus chunks       : {len(chunks)}")
print(f"  QA pairs evaluated  : {len(qa_set)}")
print(f"  Retriever top_k     : 3")
print(f"  Embedding model     : text-embedding-3-small")
print(f"  Generator model     : gpt-4o-mini")
print()
print(f"  Retrieval hit@3     : {ret_df['hit@k'].mean():.2%}")
print(f"  Retrieval MRR@3     : {ret_df['rr'].mean():.3f}")
print(f"  End-to-end accuracy : {gen_df['correct'].mean():.2%}")
print("=" * 60)

RAG pipeline evaluation summary
  Corpus chunks       : 64
  QA pairs evaluated  : 10
  Retriever top_k     : 3
  Embedding model     : text-embedding-3-small
  Generator model     : gpt-4o-mini

  Retrieval hit@3     : 90.00%
  Retrieval MRR@3     : 0.667
  End-to-end accuracy : 90.00%


## 11. Live Q&A — ask anything about the paper

In [12]:
questions = [
    "What is multi-head attention in the Transformer model?",
    "Why do the authors use scaled dot-product attention?",
    "How many encoder and decoder layers does the base Transformer have?",
    "What is positional encoding and why is it needed?",
    "What datasets did they evaluate on?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}")
    print("-" * 70)

Q: What is multi-head attention in the Transformer model?
A: Multi-head attention in the Transformer model allows the model to jointly attend to information from different representation subspaces at different positions. It consists of multiple parallel attention layers (or heads), where each head computes attention using different projections of the input queries, keys, and values. The outputs of these heads are concatenated and projected to form the final values. This approach enables the model to capture diverse information and relationships in the data, enhancing its performance compared to single-head attention.
----------------------------------------------------------------------
Q: Why do the authors use scaled dot-product attention?
A: The authors use scaled dot-product attention to counteract the effect of large dot products that can push the softmax function into regions with extremely small gradients, which can hinder learning. By scaling the dot products by \( \frac{1}{\sq

## 12. (Optional) Configuration sweep — manual AutoRAG-style comparison

The whole point of AutoRAG is comparing configurations. Here's the same idea in 30 lines: try different (chunk_size, k) combinations and rank them on retrieval hit-rate. No external framework needed.

In [13]:
configs = [
    {"chunk_size": 400, "overlap": 50, "k": 3},
    {"chunk_size": 800, "overlap": 100, "k": 3},
    {"chunk_size": 800, "overlap": 100, "k": 5},
    {"chunk_size": 1200, "overlap": 150, "k": 5},
]

sweep_results = []
for cfg in configs:
    sp = RecursiveCharacterTextSplitter(
        chunk_size=cfg["chunk_size"],
        chunk_overlap=cfg["overlap"],
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    cks = sp.split_documents(pages)
    for i, c in enumerate(cks):
        c.metadata["chunk_id"] = i

    vs = Chroma.from_documents(
        documents=cks,
        embedding=embeddings,
        collection_name=f"sweep_{cfg['chunk_size']}_{cfg['k']}",
    )

    # Re-run QA generation for THIS chunk set so gt_chunk_id stays valid
    random.seed(0)
    sweep_qa = []
    for i, ck in enumerate(random.sample(cks, min(n_qa, len(cks)))):
        raw = qa_chain.invoke({"passage": ck.page_content}).strip()
        raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        try:
            parsed = json.loads(raw)
            sweep_qa.append({"question": parsed["question"], "gt_chunk_id": ck.metadata["chunk_id"]})
        except Exception:
            pass

    hits = []
    rrs = []
    for qa in sweep_qa:
        retrieved = vs.similarity_search(qa["question"], k=cfg["k"])
        ids = [d.metadata["chunk_id"] for d in retrieved]
        gt = qa["gt_chunk_id"]
        hit = gt in ids
        hits.append(hit)
        rrs.append(1.0 / (ids.index(gt) + 1) if hit else 0.0)

    sweep_results.append({
        **cfg,
        "n_chunks": len(cks),
        "hit_rate": sum(hits) / len(hits) if hits else 0,
        "mrr": sum(rrs) / len(rrs) if rrs else 0,
    })

sweep_df = pd.DataFrame(sweep_results).sort_values("hit_rate", ascending=False)
print(sweep_df.to_string(index=False))
print(f"\nBest config: {sweep_df.iloc[0].to_dict()}")

 chunk_size  overlap  k  n_chunks  hit_rate      mrr
       1200      150  5        41       1.0 0.883333
        800      100  5        64       1.0 0.741667
        800      100  3        64       0.9 0.716667
        400       50  3       119       0.9 0.850000

Best config: {'chunk_size': 1200.0, 'overlap': 150.0, 'k': 5.0, 'n_chunks': 41.0, 'hit_rate': 1.0, 'mrr': 0.8833333333333334}
